In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **So lets begin with complete EDA**

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# **Load Data**

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [ ]:
print("Train Shape: ",train.shape)
print("Test Shape: ",test.shape)

In [ ]:
print(train.info())

In [ ]:
print(test.info())

# **Check for NULL values Train Data**

In [ ]:
print(train.isnull().sum())

In [ ]:
sns.heatmap(train.isnull())

# **Test Data**

In [ ]:
print(test.isnull().sum())

In [ ]:
sns.heatmap(test.isnull())

# **Handling NULL data**

# **For Train Data**

In [ ]:
cat_col_train = ['FireplaceQu', 'GarageType','GarageFinish','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageQual','GarageCond']
ncat_col_train = ['LotFrontage','GarageYrBlt','MasVnrArea']
for i in cat_col_train:
    train[i] = train[i].fillna(train[i].mode()[0])

for j in ncat_col_train:
    train[j] = train[j].fillna(train[j].mean())

# **For Test Data**

In [ ]:
cat_col_test = ['FireplaceQu','GarageType','GarageFinish','MasVnrType','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','FireplaceQu','GarageQual','GarageCond','MSZoning','Utilities','Exterior1st','Exterior2nd','KitchenQual','Functional','SaleType']
ncat_col_test = ['LotFrontage','GarageYrBlt','MasVnrArea','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','GarageCars','GarageArea']
for i in cat_col_test:
    test[i] = test[i].fillna(test[i].mode()[0])
for j in ncat_col_test:
    test[j] = test[j].fillna(test[j].mean())

# **Drop Columns**

In [ ]:
to_drop = ['Id','Alley','PoolQC','Fence','MiscFeature']
for k in to_drop:
    train.drop([k], axis = 1, inplace = True)
    test.drop([k], axis = 1, inplace = True)

In [ ]:
sns.heatmap(train.isnull())

In [ ]:
sns.heatmap(test.isnull())

In [ ]:
print("Train Shape: ",train.shape)
print("Test Shape: ",test.shape)

In [ ]:
final_df = pd.concat([train,test], axis = 0)
final_df.shape

In [ ]:
all_cat_col = ['MSZoning','Street','LotShape','LandContour','Utilities','LotConfig','LandSlope','Neighborhood','Condition1','Condition2','BldgType','HouseStyle','RoofStyle','RoofMatl','Exterior1st','Exterior2nd','MasVnrType','ExterQual','ExterCond','Foundation','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','Heating','HeatingQC','CentralAir','Electrical','KitchenQual','Functional','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','PavedDrive','SaleType','SaleCondition']
def cat_onehot_encoding(multicol):
    df_final = final_df
    i = 0
    for fields in multicol:
        print(fields)
        df1 = pd.get_dummies(final_df[fields], drop_first = True)

        final_df.drop([fields], axis = 1, inplace = True)
        if i==0:
            df_final = df1.copy()
        else:
            df_final = pd.concat([df_final,df1], axis = 1)
        i = i + 1
    df_final = pd.concat([final_df, df_final], axis = 1)
    return df_final
final_df = cat_onehot_encoding(all_cat_col)
        

In [ ]:
final_df.shape

In [ ]:
final_df = final_df.loc[:,~final_df.columns.duplicated()]
final_df.shape

In [ ]:
df_train = final_df.iloc[:1460,:]
df_test = final_df.iloc[1460:,:]

df_test.drop(['SalePrice'], axis = 1, inplace = True)

print("Train Shape: ",df_train.shape)
print("Test Shape: ", df_test.shape)

# **Training Data**

In [ ]:
x_train = df_train.drop(['SalePrice'], axis = 1)
y_train = df_train['SalePrice']

# **XGBoost**

In [ ]:
import xgboost
xgb_model = xgboost.XGBRegressor()
xgb_model.fit(x_train, y_train)

In [ ]:
param = {
    'n_estimators': [100, 500, 900, 1100, 1500],
    'max_depth': [2,3,5,10,15],
    'learning_rate': [0.05, 0.1, 0.15, 0.2],
    'min_child_weight': [1,2,3,4],
    'booster': ['gbtree', 'gblinear'],
    'base_score': [0.25, 0.5, 0.75, 1]
}
xgb_model = xgboost.XGBRegressor(verbosity=0)
from sklearn.model_selection import RandomizedSearchCV

random_cv = RandomizedSearchCV(estimator = xgb_model,
                             param_distributions = param,
                             cv = 5, n_iter = 50,
                             scoring = 'neg_mean_absolute_error', n_jobs = 4,
                             verbose = 5,
                             return_train_score = True,
                             random_state = 42)
import warnings
warnings.filterwarnings('ignore')
random_cv.fit(x_train, y_train)

# **Save Model**

In [ ]:
f = "xgb_model.pkl"
pickle.dump(xgb_model, open(f, "wb"))

# **Predictions**

In [ ]:
xgb_model = random_cv.best_estimator_
pred_xgb = xgb_model.predict(df_test)
print(pred_xgb.shape)

# **Submission**

In [ ]:
sub_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv')
sub_df['SalePrice'] = pred_xgb
sub_df.to_csv('sample_sub_xgb.csv', index = False)

# **Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier()
dt_model.fit(x_train, y_train)

In [ ]:
print(dt_model.get_params())

# **Predictions**

In [ ]:
pred_dt = dt_model.predict(df_test)
print(pred_dt.shape)

# **Submission**

In [ ]:
sub_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv')
sub_df['SalePrice'] = pred_dt
sub_df.to_csv('sample_sub_dt.csv', index = False)

In [ ]:
x_train_numeric = np.asarray(x_train).astype('float32')
y_train_numeric = np.asarray(y_train).astype('float32')

# **Artificial Neural Network**

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from keras import ops

def root_mean_squared_error(y_true, y_pred):
    return ops.sqrt(ops.mean(ops.square(y_pred - y_true)))


# **Model**

In [ ]:
nn_model = Sequential()
nn_model.add(Dense(50, kernel_initializer = 'he_uniform', activation = 'relu', input_dim = x_train.shape[1]))
nn_model.add(Dense(25, kernel_initializer = 'he_uniform', activation = 'relu'))
nn_model.add(Dense(50, kernel_initializer = 'he_uniform', activation = 'relu'))
nn_model.add(Dense(1, kernel_initializer = 'he_uniform'))

nn_model.compile(loss = root_mean_squared_error, optimizer = "Adamax")

nn_model.fit(x_train_numeric, y_train_numeric, validation_split = 0.25, batch_size = 10, epochs = 1000)

# **Predictions**

In [ ]:
pred_nn = nn_model.predict(df_test)
print(pred_nn.shape)

# **Submission**

In [ ]:
sub_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv')
sub_df['SalePrice'] = pred_nn
sub_df.to_csv('submission.csv', index = False)